# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: Exploration with `mlcroissant`

This notebook demonstrates how to explore, extract, and process the FAIR² dataset using the `mlcroissant` Python library.

### Dataset Source
The dataset is accessed using a Croissant schema at the following URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id` identifiers using the Croissant metadata loaded above.

In [ ]:
# List available record sets and their fields by @id
record_sets = []
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        record_sets.append(rs['@id'])
        print(f"RecordSet @id: {rs['@id']} | name: {rs.get('name', None)}")
        fields = rs.get('fields', [])
        for f in fields:
            print(f"  Field @id: {f['@id']} | name: {f.get('name', None)}")
else:
    print("No explicit record sets listed in the metadata. Attempting autodetection by sampling records...")
    # Try to enumerate record set names from the dataset
    try:
        available = dataset._data.ontology.record_sets  # fallback hack
        for rs in available:
            print(f"RecordSet @id: {rs['@id']} | name: {rs.get('name', None)}")
            record_sets.append(rs['@id'])
            if 'fields' in rs:
                for f in rs['fields']:
                    print(f"  Field @id: {f['@id']} | name: {f.get('name', None)}")
    except Exception as err:
        print(f"Unable to auto-detect record sets: {err}")
        print("You may need to inspect dataset or refer to dataset documentation for correct record set IDs.")

# If record sets are not found by metadata, try to infer typical tabular records (standard Croissant table) @id
if not record_sets:
    # Try a guess based on the dataset structure
    print("Trying a generic record set name common in Croissant datasets...")
    record_sets = ['https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json#ClinicalData'] # example fallback; edit as required
    print(f"Using fallback record_set @id: {record_sets[0]}")

## 2a. Preview Records from a Record Set

Preview sample records from one record set using its `@id`. Replace `<record_set_id>` with one of the IDs listed above.

In [ ]:
# Pick a record set @id -- update this based on the previous cell output
if len(record_sets) > 0:
    record_set_id = record_sets[0]
else:
    raise RuntimeError('No record sets discovered!')

# Preview sample records by @id
n_to_preview = 3
for i, rec in enumerate(dataset.records(record_set=record_set_id)):
    print(rec)
    if i >= n_to_preview - 1:
        break

## 3. Data Extraction

Load data from each record set into a DataFrame for analysis, referencing the record set and field `@id` values from the above overview.

In [ ]:
# Prepare to load all discovered record sets into DataFrames
import collections

dataframes = collections.OrderedDict()
# Only keep non-empty record set IDs
for rsid in record_sets:
    print(f"Loading record set: {rsid}")
    records = list(dataset.records(record_set=rsid))
    if len(records) > 0:
        dataframes[rsid] = pd.DataFrame(records)
        print(f"  Columns: {dataframes[rsid].columns.tolist()}")
        print(f"  Loaded {len(dataframes[rsid])} records.")
    else:
        print(f"  No records found in this set.")

# Example: Show first 5 rows and field (column) names from the main record set
main_record_set = list(dataframes.keys())[0]
print("\nAvailable columns (@id):")
print(dataframes[main_record_set].columns.tolist())
dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)

Process the extracted DataFrame: filtering, normalization, and grouping by attribute. Here, all fields are referenced by their Croissant `@id`.

- Choose a numeric field `@id` (for example, one describing an interval or age) and a group field (such as sex, diagnosis, or anatomical location `@id`).

In [ ]:
# Identify potential numeric and group fields based on columns
df = dataframes[main_record_set]

# Attempt to auto-infer numeric and group fields
potential_numeric_fields = [col for col in df.columns if any(word in col.lower() for word in ['interval', 'age', 'count', 'years', 'number'])]
potential_group_fields = [col for col in df.columns if any(word in col.lower() for word in ['sex', 'gender', 'site', 'location', 'diagnosis', 'group', 'type'])]

print(f"Potential numeric fields: {potential_numeric_fields}")
print(f"Potential group fields: {potential_group_fields}")

# For this example, select the first available numeric and group field @id
if len(potential_numeric_fields) == 0 or len(potential_group_fields) == 0:
    print("ERROR: Unable to automatically identify numeric or group field. Please edit the notebook with actual @id values from your data.")
else:
    numeric_field_id = potential_numeric_fields[0]  # E.g. '@id' for 'interval_years' or similar
    group_field_id = potential_group_fields[0]      # E.g. '@id' for 'sex' or 'tumor_location', etc.
    print(f"Using numeric field @id: {numeric_field_id}")
    print(f"Using group field @id: {group_field_id}")

    # Convert to numeric if needed
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Filter: Only rows with the numeric value > threshold
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:\n", filtered_df[[numeric_field_id, group_field_id]].head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the group_field and show means
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization

Visualize the distribution of the numeric field and summarize means by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only run visualization if previous EDA has selected fields
if 'numeric_field_id' in locals() and 'group_field_id' in locals():
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

    # Barplot for group means if group_field exists
    if group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().dropna()
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, you learned how to:
- Load and inspect a FAIR²-compliant Croissant dataset using the `mlcroissant` library
- Reference and extract records via their Croissant `@id` for full transparency
- Perform simple filtering, normalization, grouping, and visualizations on selected numeric and categorical fields

For further analysis, adjust field and record set `@id` values to target additional or specific variables documented in the Croissant schema. Always consult field definitions and dataset documentation for appropriate use.